# LeetCode 426 — Convert Binary Search Tree to Sorted Doubly Linked List bằng Python

Link đề: https://leetcode.com/problems/convert-binary-search-tree-to-sorted-doubly-linked-list/ · Độ khó: **Medium** · Chủ đề: Tree, Binary Search Tree, Stack, DFS, Linked List, Doubly-Linked List

Skeleton LeetCode:

```python
"""
# Definition for a Node.
class Node:
    def __init__(self, val, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right
"""

class Solution:
    def treeToDoublyList(self, root: 'Optional[Node]') -> 'Optional[Node]':
        ...
```

Ghi chú: đây là bài **Premium** (🔒). Đề trong notebook được viết lại theo hiểu biết về bài; nếu bạn có tài khoản Premium, hãy **đối chiếu lại đề gốc** (nhất là cách LeetCode in kết quả, và hai ví dụ). Ví dụ nào ghi "tự dựng" là do mình thêm để minh họa. Mọi lời giải đã được kiểm chứng bằng code (kiểm tra vòng tròn đôi + đối chiếu nhiều cách trên cây ngẫu nhiên).

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: biến **cây tìm kiếm nhị phân (BST)** thành **danh sách liên kết đôi dạng vòng, đã sắp xếp** — **tại chỗ** (in place), tức là **tái sử dụng chính các nút cũ**, dùng `left` làm "con trỏ về phần tử đứng trước" (`prev`) và `right` làm "con trỏ tới phần tử đứng sau" (`next`); trả về nút **nhỏ nhất**.
2. Nắm các công cụ: lớp `Node`, dựng cây từ list kiểu LeetCode, **duyệt trung thứ tự (inorder)** của BST ra dãy tăng dần, từ khóa **`nonlocal`**, chỉ số âm `nodes[-1]`, toán tử `%` để "quấn vòng", và cách **kiểm tra một danh sách vòng đôi** có hợp lệ không.
3. Đi từ cách **gom nút vào list rồi nối lại** → **inorder đệ quy + con trỏ `prev`** → **chia để trị: nối hai vòng con + nút gốc** → **inorder lặp bằng ngăn xếp**.
4. Hiểu mấu chốt: inorder của BST đi qua các nút **theo đúng thứ tự tăng dần**, nên chỉ cần nối **nút vừa thăm** với **nút thăm ngay trước nó** (`prev`), rồi **khép vòng** bằng cách nối nút đầu với nút cuối.
5. Tránh các lỗi kinh điển: quên khép vòng, quên `nonlocal`, nối con trỏ **trước khi** đi xuống con trái (làm hỏng cây đang duyệt), trả về `root` thay vì nút nhỏ nhất, quên trường hợp cây rỗng.

## 2. Đề bài & ví dụ

Cho gốc `root` của một **cây tìm kiếm nhị phân**. Hãy chuyển nó thành **danh sách liên kết đôi vòng tròn đã sắp xếp**, **tại chỗ**:

- Với mỗi nút, con trỏ **`left`** phải trỏ tới nút **đứng trước** nó (phần tử nhỏ hơn liền kề), còn **`right`** trỏ tới nút **đứng sau** (phần tử lớn hơn liền kề).
- Vì là danh sách **vòng**: phần tử **đầu** (nhỏ nhất) có `left` trỏ tới phần tử **cuối** (lớn nhất), và phần tử cuối có `right` trỏ về phần tử **đầu**.
- Trả về con trỏ tới **phần tử nhỏ nhất**. Nếu cây rỗng thì trả về `None`.

**Ví dụ 1:** `root = [4,2,5,1,3]` (cây có gốc 4, con trái 2, con phải 5; 2 có con 1 và 3). Kết quả là vòng `1 ⇄ 2 ⇄ 3 ⇄ 4 ⇄ 5 ⇄ (về 1)`; LeetCode in ra `[1,2,3,4,5]` (đọc theo `right` từ nút nhỏ nhất).

**Ví dụ 2:** `root = [2,1,3]` → vòng `1 ⇄ 2 ⇄ 3 ⇄ (về 1)`, in `[1,2,3]`.

**Ràng buộc:** số nút trong `[0, 2000]`; `-1000 <= Node.val <= 1000`; mọi giá trị **khác nhau đôi một**.

(Vì đề dùng đối tượng `Node` và con trỏ, trong notebook mình **tự dựng** `Node`, hàm dựng cây, và hàm kiểm tra vòng đôi ở mục 3.)

In [1]:
examples = [
    ([4, 2, 5, 1, 3], [1, 2, 3, 4, 5]),
    ([2, 1, 3], [1, 2, 3]),
    ([], []),
]
for lst, expected in examples:
    print("root =", lst, "-> danh sách kỳ vọng (đọc theo right từ nút nhỏ nhất):", expected)

root = [4, 2, 5, 1, 3] -> danh sách kỳ vọng (đọc theo right từ nút nhỏ nhất): [1, 2, 3, 4, 5]
root = [2, 1, 3] -> danh sách kỳ vọng (đọc theo right từ nút nhỏ nhất): [1, 2, 3]
root = [] -> danh sách kỳ vọng (đọc theo right từ nút nhỏ nhất): []


## 3. Kiến thức nền

### 3.1. Nút cây `Node` — một lớp, hai cách đọc

Một nút có `val`, `left`, `right`. **Trong cây**, `left`/`right` là **con trái/con phải**. **Trong danh sách đôi**, *cùng hai thuộc tính đó* là **`prev`/`next`**. Bài này chính là "đổi nghĩa" hai con trỏ mà không tạo nút mới.

In [2]:
class Node:
    def __init__(self, val, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right

    def __repr__(self):
        return f"Node({self.val})"


a = Node(2, Node(1), Node(3))
print(a, a.left, a.right)      # Node(2) Node(1) Node(3)
print(a.left.left)             # None: nút 1 không có con trái

Node(2) Node(1) Node(3)
None


### 3.2. Dựng cây từ list kiểu LeetCode: `build_tree`

LeetCode viết cây theo **thứ tự từng tầng** (level order), `None` là chỗ trống. `[4,2,5,1,3]` nghĩa là: gốc `4`; hai con của `4` là `2`, `5`; hai con của `2` là `1`, `3`. Ta dùng một **hàng đợi**: lấy nút cha ra, đọc 2 giá trị kế tiếp làm con trái/phải.

In [3]:
from collections import deque

def build_tree(vals):
    if not vals or vals[0] is None:
        return None
    root = Node(vals[0])
    q = deque([root])
    i = 1
    while q and i < len(vals):
        node = q.popleft()
        if i < len(vals) and vals[i] is not None:
            node.left = Node(vals[i]); q.append(node.left)
        i += 1
        if i < len(vals) and vals[i] is not None:
            node.right = Node(vals[i]); q.append(node.right)
        i += 1
    return root


def show_tree(node, depth=0):
    """In cây nằm ngang: phải ở trên, trái ở dưới (xoay trái 90 độ)."""
    if node is None:
        return
    show_tree(node.right, depth + 1)
    print("    " * depth + str(node.val))
    show_tree(node.left, depth + 1)


t = build_tree([4, 2, 5, 1, 3])
show_tree(t)

    5
4
        3
    2
        1


### 3.3. BST và thứ tự **trung tự (inorder)**

Một BST thỏa: mọi giá trị ở cây con trái **nhỏ hơn** nút, mọi giá trị ở cây con phải **lớn hơn**. Duyệt **trái → nút → phải** (inorder) thì các giá trị ra **tăng dần**. Đây là chìa khóa của bài: danh sách phải được sắp xếp, mà inorder đã cho thứ tự sắp xếp miễn phí.

In [4]:
def inorder_vals(node):
    if node is None:
        return []
    return inorder_vals(node.left) + [node.val] + inorder_vals(node.right)

print(inorder_vals(build_tree([4, 2, 5, 1, 3])))     # [1, 2, 3, 4, 5]
print(inorder_vals(build_tree([2, 1, 3])))           # [1, 2, 3]

[1, 2, 3, 4, 5]
[1, 2, 3]


### 3.4. Danh sách liên kết đôi vòng tròn

- **Đôi:** mỗi nút có `prev` và `next`, đi tới/lui được.
- **Vòng:** không có `None` ở hai đầu. `tail.next = head` và `head.prev = tail`.
- Hệ quả: một danh sách **một nút** là vòng tự trỏ vào chính nó: `node.left = node.right = node`.

Hãy tự nối tay danh sách vòng `1 ⇄ 2 ⇄ 3` để thấy.

In [5]:
n1, n2, n3 = Node(1), Node(2), Node(3)
n1.right, n2.left = n2, n1          # 1 <-> 2
n2.right, n3.left = n3, n2          # 2 <-> 3
n3.right, n1.left = n1, n3          # khép vòng: 3 -> 1 và 1 <- 3

cur = n1
for _ in range(7):                  # đi vòng hơn một lần để thấy nó quay lại
    print(cur.val, end=" -> ")
    cur = cur.right
print("...")
print("lùi từ 1:", n1.left.val, n1.left.left.val)    # 3 rồi 2

1 -> 2 -> 3 -> 1 -> 2 -> 3 -> 1 -> ...
lùi từ 1: 3 2


### 3.5. `nonlocal` — sửa biến của hàm bao ngoài

Trong hàm đệ quy lồng bên trong, nếu viết `prev = root` thì Python hiểu `prev` là **biến cục bộ mới**. Muốn gán vào biến `prev` của hàm ngoài phải khai báo `nonlocal prev`. (Chỉ **đọc** thì không cần; chỉ **gán lại** mới cần.)

In [6]:
def demo_nonlocal():
    count = 0
    def add():
        nonlocal count        # thử bỏ dòng này sẽ lỗi UnboundLocalError
        count += 1
    add(); add(); add()
    return count

print(demo_nonlocal())        # 3

3


### 3.6. Chỉ số âm và `%` để "quấn vòng"

`nodes[-1]` là **phần tử cuối**. Còn `(i + 1) % n` quay về `0` khi `i = n - 1`. Hai mẹo này cho phép nối vòng bằng **một công thức duy nhất** cho mọi `i`, kể cả hai đầu.

In [7]:
nodes = ["a", "b", "c", "d"]
n = len(nodes)
for i in range(n):
    print(f"i={i}: trước = nodes[{i-1}] = {nodes[i-1]}, sau = nodes[({i}+1)%{n}] = {nodes[(i + 1) % n]}")

i=0: trước = nodes[-1] = d, sau = nodes[(0+1)%4] = b
i=1: trước = nodes[0] = a, sau = nodes[(1+1)%4] = c
i=2: trước = nodes[1] = b, sau = nodes[(2+1)%4] = d
i=3: trước = nodes[2] = c, sau = nodes[(3+1)%4] = a


### 3.7. Hàm kiểm tra vòng đôi `check_circular`

Vì bài có **kết quả là cấu trúc con trỏ**, ta cần một "giám khảo". `check_circular(head, expected)` xác nhận: (1) đi `right` đúng `n` bước thì ra đúng dãy `expected` và **quay về `head`**; (2) mọi nút thỏa `x.right.left is x`; (3) đi `left` ngược từ `head` ra dãy đảo. Có **bảo vệ** để không lặp vô hạn khi cấu trúc sai.

In [8]:
def check_circular(head, expected):
    n = len(expected)
    if n == 0:
        return (head is None, "rỗng" if head is None else "đáng lẽ phải None")
    if head is None:
        return False, "head là None"
    cur, fwd, seen = head, [], set()
    for _ in range(n):
        if cur is None:
            return False, "right bị None giữa chừng (chưa thành vòng)"
        if id(cur) in seen:
            return False, "vòng khép quá sớm"
        seen.add(id(cur)); fwd.append(cur.val)
        if cur.right is None or cur.right.left is not cur:
            return False, f"right/left không đối xứng tại nút {cur.val}"
        cur = cur.right
    if cur is not head:
        return False, "đi đủ n bước nhưng không quay về head"
    if fwd != expected:
        return False, f"thứ tự sai: {fwd}"
    cur, back = head, []
    for _ in range(n):
        cur = cur.left
        back.append(cur.val)
    if back != expected[::-1]:
        return False, f"đi ngược sai: {back}"
    return True, "ok"


def circle_list(head, limit=None):
    """Đọc vòng bằng right từ head (dừng khi quay về head) — chỉ để in."""
    if head is None:
        return []
    out, cur = [head.val], head.right
    while cur is not head and cur is not None and len(out) < 5000:
        out.append(cur.val); cur = cur.right
    return out

print(check_circular(n1, [1, 2, 3]))          # (True, 'ok')   vòng tay nối ở 3.4
print(check_circular(n1, [1, 2, 4]))          # sai dãy

(True, 'ok')
(False, 'thứ tự sai: [1, 2, 3]')


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Cây `[4,2,5,1,3]`:

```
      4
     / \
    2   5
   / \
  1   3
```

Inorder: `1, 2, 3, 4, 5`. Mong muốn cuối: `1 ⇄ 2 ⇄ 3 ⇄ 4 ⇄ 5` và `5.right = 1`, `1.left = 5`.

Tính tay "ai nối với ai" theo thứ tự thăm:

| Thăm | `prev` trước đó | Việc cần làm |
|---|---|---|
| 1 | (chưa có) | `1` là **đầu danh sách**, ghi `head = 1` |
| 2 | 1 | `1.right = 2`, `2.left = 1` |
| 3 | 2 | `2.right = 3`, `3.left = 2` |
| 4 | 3 | `3.right = 4`, `4.left = 3` |
| 5 | 4 | `4.right = 5`, `5.left = 4` |
| xong | 5 | khép vòng: `5.right = 1`, `1.left = 5` |

Chỉ có **một quy tắc lặp lại**: "nối nút hiện tại với nút vừa thăm trước nó". Phần việc còn lại là làm sao **duyệt theo inorder** mà không làm hỏng cây khi ta đang sửa con trỏ.

## 5. Cách 1 — Brute Force / baseline: gom nút vào list rồi nối lại

**Ý tưởng.** Tách làm hai việc cho dễ hiểu:

1. Duyệt inorder, **bỏ các nút (đối tượng) vào một list** `nodes` → `nodes` đã sắp xếp.
2. Với mỗi `i`: `nodes[i].left = nodes[i-1]`, `nodes[i].right = nodes[(i+1) % n]` — công thức quấn vòng của mục 3.6.

Vì **đã lưu hết nút vào list trước**, việc ghi đè `left/right` sau đó không còn làm hỏng việc duyệt. Cách này dễ đúng nhất, nên làm baseline/oracle.

In [9]:
def tree_to_dll_list(root):
    if root is None:
        return None
    nodes = []

    def inorder(node):
        if node is None:
            return
        inorder(node.left)
        nodes.append(node)          # lưu CHÍNH nút (đối tượng), không phải giá trị
        inorder(node.right)

    inorder(root)
    n = len(nodes)
    for i, node in enumerate(nodes):
        node.left = nodes[i - 1]            # i = 0 -> nodes[-1] (phần tử cuối)
        node.right = nodes[(i + 1) % n]     # i = n-1 -> nodes[0]
    return nodes[0]


for lst, expected in examples:
    head = tree_to_dll_list(build_tree(lst))
    print(lst, "->", circle_list(head), check_circular(head, expected))

[4, 2, 5, 1, 3] -> [1, 2, 3, 4, 5] (True, 'ok')
[2, 1, 3] -> [1, 2, 3] (True, 'ok')
[] -> [] (True, 'rỗng')


**Độ phức tạp (Cách 1).** Gọi `n` là số nút, `h` là chiều cao. Time **`O(n)`**. Space **`O(n)`** cho list `nodes` (cộng `O(h)` ngăn xếp đệ quy).

**Nút thắt:** đề nói "tại chỗ" — dù ta không tạo nút mới, ta vẫn tốn thêm `O(n)` bộ nhớ phụ cho list. Có thể chỉ cần nhớ **một** nút: nút vừa thăm (`prev`).

## 6. Ý tưởng thuật toán / bước đột phá

Từ bảng mục 4: để nối nút hiện tại `cur`, ta chỉ cần **nút thăm ngay trước đó** (`prev`). Vậy thay vì lưu cả list, giữ **một biến `prev`** và **một biến `head`** (nút thăm đầu tiên = nhỏ nhất) trong lúc duyệt inorder:

- Khi thăm `cur`: nếu `prev` chưa có → `head = cur`; ngược lại `prev.right = cur; cur.left = prev`.
- Sau đó `prev = cur`.
- Duyệt xong: `prev` chính là nút lớn nhất → **khép vòng** `prev.right = head; head.left = prev`.

**Vì sao an toàn khi sửa con trỏ lúc đang duyệt?** Ta sửa `cur.left` và `prev.right` **sau khi** đã đi hết cây con trái của `cur` (inorder: trái xong mới thăm nút), và `prev` là nút đã thăm xong (không còn cần `prev.right` cũ vì đã đi sang phải của nó rồi). Còn `cur.right` **chưa bị đụng** nên vẫn đi tiếp xuống cây con phải được. Thứ tự "trái → nút → phải" chính là thứ làm mọi thứ khớp.

## 7. Các cách tiếp theo

### Cách 2 — Inorder đệ quy + con trỏ `prev` (lời giải chuẩn)

**Giải thích từng đoạn.**

1. `dfs(node)`: nếu `node is None` thì thoát.
2. `dfs(node.left)` — thăm hết các nút nhỏ hơn trước.
3. **Thăm `node`**: nếu `prev` có rồi thì nối hai chiều với `node`; nếu chưa thì đây là nút nhỏ nhất → `head = node`. Rồi `prev = node`.
4. `dfs(node.right)` — thăm các nút lớn hơn.
5. Sau `dfs(root)`: khép vòng giữa `prev` (cuối) và `head` (đầu).

In [10]:
def tree_to_dll_inorder(root):
    if root is None:
        return None
    head = prev = None

    def dfs(node):
        nonlocal head, prev            # sẽ GÁN lại hai biến này -> cần nonlocal
        if node is None:
            return
        dfs(node.left)
        if prev is not None:
            prev.right = node
            node.left = prev
        else:
            head = node                # nút đầu tiên được thăm = nhỏ nhất
        prev = node
        dfs(node.right)

    dfs(root)
    prev.right = head                   # khép vòng
    head.left = prev
    return head


for lst, expected in examples:
    head = tree_to_dll_inorder(build_tree(lst))
    print(lst, "->", circle_list(head), check_circular(head, expected))

[4, 2, 5, 1, 3] -> [1, 2, 3, 4, 5] (True, 'ok')
[2, 1, 3] -> [1, 2, 3] (True, 'ok')
[] -> [] (True, 'rỗng')


In [11]:
class Solution2:
    def treeToDoublyList(self, root: 'Optional[Node]') -> 'Optional[Node]':
        if root is None:
            return None
        head = prev = None

        def dfs(node):
            nonlocal head, prev
            if node is None:
                return
            dfs(node.left)
            if prev is not None:
                prev.right = node
                node.left = prev
            else:
                head = node
            prev = node
            dfs(node.right)

        dfs(root)
        prev.right = head
        head.left = prev
        return head


head = Solution2().treeToDoublyList(build_tree([4, 2, 5, 1, 3]))
print(circle_list(head), check_circular(head, [1, 2, 3, 4, 5]))

[1, 2, 3, 4, 5] (True, 'ok')


**Độ phức tạp (Cách 2).** Time **`O(n)`** (mỗi nút thăm đúng một lần). Space **`O(h)`** cho ngăn xếp đệ quy (`h = log n` khi cây cân bằng, `h = n` khi cây lệch thành chuỗi) và **`O(1)` bộ nhớ phụ ngoài ngăn xếp**.

### Cách 3 — Chia để trị: nối "vòng con trái" + nút gốc + "vòng con phải"

Đây là một ý tưởng **khác hẳn**: không cần biến `prev` toàn cục. Coi hàm `convert(node)` làm đúng đề cho **cây con**: trả về đầu của **danh sách vòng đã sắp xếp** của cây con đó. Khi đó:

1. `L = convert(node.left)`, `R = convert(node.right)` — hai vòng, mọi giá trị trong `L` < `node` < mọi giá trị trong `R`.
2. Biến `node` thành **vòng một phần tử** (`node.left = node.right = node`).
3. Nối `L + node + R` bằng hàm `concat(a, b)` nối **hai vòng** thành một vòng trong `O(1)`: lấy `a_tail = a.left`, `b_tail = b.left`; rồi `a_tail.right = b; b.left = a_tail; b_tail.right = a; a.left = b_tail`.

Thứ tự `L, node, R` đảm bảo đã sắp xếp. Nếu một bên là `None` thì `concat` trả về bên kia.

In [12]:
def concat(a, b):
    """Nối hai danh sách vòng (đầu a, đầu b) thành một vòng, trả về đầu a. O(1)."""
    if a is None:
        return b
    if b is None:
        return a
    a_tail, b_tail = a.left, b.left
    a_tail.right, b.left = b, a_tail      # cuối a -> đầu b
    b_tail.right, a.left = a, b_tail      # cuối b -> đầu a (khép vòng mới)
    return a


def tree_to_dll_divide(root):
    if root is None:
        return None
    left_ring = tree_to_dll_divide(root.left)
    right_ring = tree_to_dll_divide(root.right)
    root.left = root.right = root          # vòng một nút
    return concat(concat(left_ring, root), right_ring)


for lst, expected in examples:
    head = tree_to_dll_divide(build_tree(lst))
    print(lst, "->", circle_list(head), check_circular(head, expected))

[4, 2, 5, 1, 3] -> [1, 2, 3, 4, 5] (True, 'ok')
[2, 1, 3] -> [1, 2, 3] (True, 'ok')
[] -> [] (True, 'rỗng')


In [13]:
class Solution3:
    def treeToDoublyList(self, root: 'Optional[Node]') -> 'Optional[Node]':
        def concat(a, b):
            if a is None:
                return b
            if b is None:
                return a
            a_tail, b_tail = a.left, b.left
            a_tail.right, b.left = b, a_tail
            b_tail.right, a.left = a, b_tail
            return a

        def convert(node):
            if node is None:
                return None
            left_ring = convert(node.left)
            right_ring = convert(node.right)
            node.left = node.right = node
            return concat(concat(left_ring, node), right_ring)

        return convert(root)


head = Solution3().treeToDoublyList(build_tree([4, 2, 5, 1, 3]))
print(circle_list(head), check_circular(head, [1, 2, 3, 4, 5]))

[1, 2, 3, 4, 5] (True, 'ok')


**Độ phức tạp (Cách 3).** Mỗi nút gọi `concat` hai lần, mỗi lần `O(1)` ⇒ Time **`O(n)`**. Space **`O(h)`** ngăn xếp đệ quy. Ưu điểm: **không cần biến "toàn cục"** (`prev`, `head`), và hàm tự đủ nghĩa ("biến cây con thành vòng"); nhược điểm: khó hình dung hơn cho người mới.

### Cách 4 — Inorder **lặp** bằng ngăn xếp (không đệ quy)

Python mặc định chỉ cho đệ quy sâu khoảng **1000** tầng, mà cây có thể có **2000** nút xếp thành chuỗi (BST lệch) ⇒ đệ quy có thể lỗi `RecursionError` (LeetCode nâng giới hạn nên thường vẫn qua, nhưng ở máy bạn thì không). Cách an toàn: tự quản ngăn xếp.

**Giải thích từng đoạn.**

1. `cur` là nút đang xét; `stack` giữ các nút "đang chờ thăm".
2. Đi hết xuống **trái** nhất, đẩy mọi nút vào `stack`.
3. `pop` ra một nút = nút nhỏ nhất chưa thăm → **thăm** (nối với `prev`).
4. Chuyển sang `cur = node.right` (đã đọc `right` **trước khi** nối nên vẫn nguyên) và lặp lại.

In [14]:
def tree_to_dll_iterative(root):
    if root is None:
        return None
    stack = []
    cur = root
    head = prev = None
    while cur is not None or stack:
        while cur is not None:          # đi hết xuống trái
            stack.append(cur)
            cur = cur.left
        node = stack.pop()              # nút nhỏ nhất còn lại
        if prev is not None:
            prev.right = node
            node.left = prev
        else:
            head = node
        prev = node
        cur = node.right                # right của node CHƯA bị sửa
    prev.right = head
    head.left = prev
    return head


for lst, expected in examples:
    head = tree_to_dll_iterative(build_tree(lst))
    print(lst, "->", circle_list(head), check_circular(head, expected))

[4, 2, 5, 1, 3] -> [1, 2, 3, 4, 5] (True, 'ok')
[2, 1, 3] -> [1, 2, 3] (True, 'ok')
[] -> [] (True, 'rỗng')


In [15]:
class Solution4:
    def treeToDoublyList(self, root: 'Optional[Node]') -> 'Optional[Node]':
        if root is None:
            return None
        stack, cur = [], root
        head = prev = None
        while cur is not None or stack:
            while cur is not None:
                stack.append(cur)
                cur = cur.left
            node = stack.pop()
            if prev is not None:
                prev.right = node
                node.left = prev
            else:
                head = node
            prev = node
            cur = node.right
        prev.right = head
        head.left = prev
        return head


head = Solution4().treeToDoublyList(build_tree([2, 1, 3]))
print(circle_list(head), check_circular(head, [1, 2, 3]))

[1, 2, 3] (True, 'ok')


**Độ phức tạp (Cách 4).** Time **`O(n)`**; Space **`O(h)`** cho `stack` (không có tầng đệ quy nên không bị giới hạn recursion). Với cây lệch, `h = n` ⇒ `O(n)`.

> Có cách thứ năm là **Morris traversal** (`O(1)` bộ nhớ thật sự) nhưng việc vừa "luồn chỉ" vừa đổi nghĩa `left/right` rất dễ rối; ở mức nhập môn, bốn cách trên là đủ.

## 8. TRACE từng bước

Trace **Cách 2** (inorder + `prev`) trên cây `[4,2,5,1,3]`, in ra từng lần "thăm" nút, `prev`, `head` và các con trỏ vừa nối.

In [16]:
def trace_tree_to_dll(root):
    head = prev = None
    step = 0

    def dfs(node, depth=0):
        nonlocal head, prev, step
        pad = "  " * depth
        if node is None:
            return
        print(f"{pad}vào {node.val}")
        dfs(node.left, depth + 1)
        step += 1
        if prev is not None:
            prev.right = node
            node.left = prev
            print(f"{pad}THĂM #{step}: {node.val}  prev={prev.val} -> nối {prev.val}.right={node.val}, {node.val}.left={prev.val}")
        else:
            head = node
            print(f"{pad}THĂM #{step}: {node.val}  prev=None -> head = {node.val}")
        prev = node
        dfs(node.right, depth + 1)

    dfs(root)
    prev.right = head
    head.left = prev
    print(f"khép vòng: {prev.val}.right = {head.val}, {head.val}.left = {prev.val}")
    return head


head = trace_tree_to_dll(build_tree([4, 2, 5, 1, 3]))
print("kết quả:", circle_list(head), check_circular(head, [1, 2, 3, 4, 5]))

vào 4
  vào 2
    vào 1
    THĂM #1: 1  prev=None -> head = 1
  THĂM #2: 2  prev=1 -> nối 1.right=2, 2.left=1
    vào 3
    THĂM #3: 3  prev=2 -> nối 2.right=3, 3.left=2
THĂM #4: 4  prev=3 -> nối 3.right=4, 4.left=3
  vào 5
  THĂM #5: 5  prev=4 -> nối 4.right=5, 5.left=4
khép vòng: 5.right = 1, 1.left = 5
kết quả: [1, 2, 3, 4, 5] (True, 'ok')


Bảng trạng thái (mỗi dòng là một lần "thăm", theo thứ tự inorder):

| Lần thăm | `node` | `prev` trước | `head` sau | Con trỏ vừa gán | `prev` sau |
|---|---|---|---|---|---|
| 1 | 1 | None | 1 | (không) | 1 |
| 2 | 2 | 1 | 1 | `1.right=2`, `2.left=1` | 2 |
| 3 | 3 | 2 | 1 | `2.right=3`, `3.left=2` | 3 |
| 4 | 4 | 3 | 1 | `3.right=4`, `4.left=3` | 4 |
| 5 | 5 | 4 | 1 | `4.right=5`, `5.left=4` | 5 |
| xong | — | 5 | 1 | `5.right=1`, `1.left=5` | — |

Chú ý nút `4` (gốc): nó chỉ được thăm **lần thứ 4** — sau khi toàn bộ cây con trái `{1,2,3}` xong. Đó là lý do `3.right` (cũ là `None`) mới được đặt thành `4` đúng lúc.

Trace thêm **Cách 4 (ngăn xếp)** để thấy `stack` thay đổi:

In [17]:
def trace_iterative(root):
    stack, cur = [], root
    prev = head = None
    while cur is not None or stack:
        while cur is not None:
            stack.append(cur)
            cur = cur.left
        print("  đẩy xong, stack =", [x.val for x in stack])
        node = stack.pop()
        action = "head" if prev is None else f"nối {prev.val}<->{node.val}"
        if prev is not None:
            prev.right = node
            node.left = prev
        else:
            head = node
        prev = node
        print(f"  pop {node.val}: {action}; sang phải -> {node.right.val if node.right else None}")
        cur = node.right
    prev.right = head
    head.left = prev
    return head


head = trace_iterative(build_tree([4, 2, 5, 1, 3]))
print(circle_list(head))

  đẩy xong, stack = [4, 2, 1]
  pop 1: head; sang phải -> None
  đẩy xong, stack = [4, 2]
  pop 2: nối 1<->2; sang phải -> 3
  đẩy xong, stack = [4, 3]
  pop 3: nối 2<->3; sang phải -> None
  đẩy xong, stack = [4]
  pop 4: nối 3<->4; sang phải -> 5
  đẩy xong, stack = [5]
  pop 5: nối 4<->5; sang phải -> None
[1, 2, 3, 4, 5]


## 9. Code nộp LeetCode

Dưới đây là bản nộp (lời giải chuẩn, Cách 2). Trên LeetCode **không** khai báo lại `Node` (đã có sẵn); trong notebook ta dùng `Node` đã định nghĩa ở mục 3.

In [18]:
class Solution:
    def treeToDoublyList(self, root: 'Optional[Node]') -> 'Optional[Node]':
        def dfs(node):
            nonlocal prev, head
            if node is None:
                return
            dfs(node.left)
            if prev:
                prev.right = node
                node.left = prev
            else:
                head = node
            prev = node
            dfs(node.right)

        if root is None:
            return None
        head = prev = None
        dfs(root)
        prev.right = head
        head.left = prev
        return head


for lst, expected in examples:
    head = Solution().treeToDoublyList(build_tree(lst))
    ok, msg = check_circular(head, expected)
    print(lst, "->", circle_list(head), ok, msg)
    assert ok

[4, 2, 5, 1, 3] -> [1, 2, 3, 4, 5] True ok
[2, 1, 3] -> [1, 2, 3] True ok
[] -> [] True rỗng


## 10. Vì sao thuật toán đúng

**Bất biến (Cách 2).** Ngay trước khi thăm một nút `cur`:

1. `prev` là nút **lớn nhất trong các nút đã thăm**, và các nút đã thăm đã tạo thành **chuỗi liên kết đôi đã sắp xếp** bắt đầu từ `head` (chưa khép vòng).
2. Tất cả nút đã thăm **nhỏ hơn** `cur` (tính chất inorder của BST).

Thăm `cur` bằng cách nối `prev ⇄ cur` thì chuỗi vẫn sắp xếp (vì `prev < cur` và `cur` là nút nhỏ nhất trong số chưa thăm) và bất biến được giữ. Khi duyệt xong, chuỗi gồm **mọi** nút, theo đúng thứ tự tăng; nối `prev ⇄ head` khép vòng.

**Vì sao sửa con trỏ không làm hỏng việc duyệt?** `prev.right` bị ghi đè nhưng `prev` đã thăm xong, ta đã đi vào cây con phải của nó rồi (hoặc nó không có). `cur.left` bị ghi đè **sau khi** `dfs(cur.left)` đã hoàn thành. `cur.right` chỉ được đọc **sau** lúc nối. Mỗi con trỏ cũ đều đã dùng xong trước khi bị ghi đè.

**Cách 3:** quy nạp theo chiều cao: nếu `convert` đúng cho hai cây con (trả vòng đã sắp xếp), thì `L + node + R` đã sắp xếp (mọi giá trị `L` < `node` < mọi giá trị `R`) và `concat` giữ tính vòng ⇒ đúng cho cả cây. **Cách 1, 4** là cùng thứ tự inorder, chỉ khác cách duyệt.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. List các nút | `O(n)` | `O(n)` |
| 2. Inorder đệ quy + `prev` | `O(n)` | `O(h)` |
| 3. Chia để trị + `concat` | `O(n)` | `O(h)` |
| 4. Inorder lặp bằng stack | `O(n)` | `O(h)` |

Với `n` = số nút, `h` = chiều cao (`log n` nếu cân bằng, `n` nếu lệch). Mỗi nút được thăm **một lần** và mỗi lần chỉ làm một số thao tác gán hằng số. Với `n ≤ 2000` mọi cách đều rất nhanh.

## 12. Bộ test

Vì thuật toán **sửa cây tại chỗ**, mỗi lần chạy một cách ta phải **dựng cây mới**. Ta cũng kiểm tra tính "tại chỗ": tập **đối tượng nút** (bằng `id`) trước và sau phải **giống hệt** (không tạo nút mới).

In [19]:
import random, sys

def collect_ids(node, out=None):
    """Thu thập id mọi nút của cây (CHỈ gọi trước khi chuyển)."""
    if out is None:
        out = set()
    stack = [node]
    while stack:
        x = stack.pop()
        if x is None:
            continue
        out.add(id(x))
        stack.append(x.left); stack.append(x.right)
    return out


def ring_ids(head):
    ids, cur = set(), head
    if head is None:
        return ids
    while True:
        ids.add(id(cur)); cur = cur.right
        if cur is head:
            return ids


def bst_from_order(order):
    """Chèn lần lượt các giá trị vào BST (dựng bằng vòng lặp, không đệ quy)."""
    root = None
    for v in order:
        if root is None:
            root = Node(v); continue
        cur = root
        while True:
            if v < cur.val:
                if cur.left is None:
                    cur.left = Node(v); break
                cur = cur.left
            else:
                if cur.right is None:
                    cur.right = Node(v); break
                cur = cur.right
    return root


ALL = {
    "cach1": tree_to_dll_list,
    "cach2": tree_to_dll_inorder,
    "cach3": tree_to_dll_divide,
    "cach4": tree_to_dll_iterative,
    "nop":   lambda r: Solution().treeToDoublyList(r),
}

def run_all(order):
    expected = sorted(order)
    for name, fn in ALL.items():
        root = bst_from_order(order)
        before = collect_ids(root)
        head = fn(root)
        ok, msg = check_circular(head, expected)
        assert ok, (name, order, msg)
        assert ring_ids(head) == before, (name, "không giữ nguyên tập nút")
    return expected

print(run_all([4, 2, 5, 1, 3]))

[1, 2, 3, 4, 5]


### 12.1. Các trường hợp biên

In [20]:
# Cây rỗng
for name, fn in ALL.items():
    assert fn(None) is None, name
print("rỗng -> None: OK")

# Một nút: vòng tự trỏ vào chính nó
for name, fn in ALL.items():
    r = Node(7)
    h = fn(r)
    assert h is r and h.left is h and h.right is h, name
print("một nút: tự trỏ chính nó: OK")

# Hai nút (con trái / con phải)
print(run_all([2, 1]), run_all([1, 2]))
# Chuỗi lệch trái, lệch phải, cây đầy đủ, giá trị âm
print(run_all([5, 4, 3, 2, 1]))
print(run_all([1, 2, 3, 4, 5]))
print(run_all([0, -1000, 1000, -500, 500]))
print("OK")

rỗng -> None: OK
một nút: tự trỏ chính nó: OK
[1, 2] [1, 2]
[1, 2, 3, 4, 5]
[1, 2, 3, 4, 5]
[-1000, -500, 0, 500, 1000]
OK


### 12.2. Phản ví dụ cho các cách làm sai

Ba hàm sai điển hình (sẽ giải thích ở mục 13). Test cho thấy `check_circular` bắt được cả ba:

In [21]:
def wrong_no_close(root):               # quên khép vòng
    if root is None:
        return None
    head = prev = None
    def dfs(node):
        nonlocal head, prev
        if node is None: return
        dfs(node.left)
        if prev is not None:
            prev.right = node; node.left = prev
        else:
            head = node
        prev = node
        dfs(node.right)
    dfs(root)
    return head                         # THIẾU: prev.right = head; head.left = prev

def wrong_return_root(root):            # trả về gốc thay vì nút nhỏ nhất
    tree_to_dll_inorder(root)
    return root

for fn in (wrong_no_close, wrong_return_root):
    ok, msg = check_circular(fn(bst_from_order([4, 2, 5, 1, 3])), [1, 2, 3, 4, 5])
    print(fn.__name__, "->", ok, "|", msg)
    assert not ok

wrong_no_close -> False | right/left không đối xứng tại nút 5
wrong_return_root -> False | thứ tự sai: [4, 5, 1, 2, 3]


### 12.3. Đối chiếu nhiều cách trên cây ngẫu nhiên (seed cố định)

Chọn ngẫu nhiên `k` giá trị **khác nhau** trong `[-1000, 1000]`, chèn theo thứ tự xáo trộn (cây có dạng ngẫu nhiên), rồi `run_all` (năm cách phải cùng được `check_circular` chấp nhận).

In [22]:
random.seed(426)
cnt = 0
for _ in range(400):
    k = random.randint(1, 60)
    order = random.sample(range(-1000, 1001), k)
    run_all(order)
    cnt += 1
print("đã đối chiếu", cnt, "cây ngẫu nhiên: OK")

đã đối chiếu 400 cây ngẫu nhiên: OK


### 12.4. Ràng buộc lớn: 2000 nút, cả cây lệch

Cây lệch 2000 nút làm đệ quy sâu 2000 tầng. Cách 4 (stack) chạy tốt; các cách đệ quy cần nâng giới hạn đệ quy (LeetCode đã làm sẵn việc này cho bạn).

In [23]:
order = list(range(-1000, 1000))                    # 2000 giá trị tăng dần -> cây lệch phải
root = bst_from_order(order)
head = tree_to_dll_iterative(root)
assert check_circular(head, sorted(order)) == (True, "ok")
print("Cách 4 (lặp) chạy 2000 nút lệch: OK")

try:
    tree_to_dll_inorder(bst_from_order(order))
    print("Cách 2 qua được ở giới hạn mặc định")
except RecursionError:
    print("Cách 2 gặp RecursionError ở giới hạn đệ quy mặc định (như dự đoán)")

old = sys.getrecursionlimit()
sys.setrecursionlimit(10000)
for fn in (tree_to_dll_inorder, tree_to_dll_divide):
    head = fn(bst_from_order(order))
    assert check_circular(head, sorted(order)) == (True, "ok")
sys.setrecursionlimit(old)
print("sau khi nâng recursion limit: OK")

random.seed(7)
order = random.sample(range(-1000, 1001), 2000)
for name, fn in ALL.items():
    sys.setrecursionlimit(10000)
    head = fn(bst_from_order(order))
    assert check_circular(head, sorted(order)) == (True, "ok"), name
sys.setrecursionlimit(old)
print("2000 nút ngẫu nhiên, mọi cách: OK")

Cách 4 (lặp) chạy 2000 nút lệch: OK


Cách 2 gặp RecursionError ở giới hạn đệ quy mặc định (như dự đoán)


sau khi nâng recursion limit: OK
2000 nút ngẫu nhiên, mọi cách: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Quên khép vòng

Danh sách **đôi** thì có, nhưng **không phải vòng**: `tail.right` vẫn `None`.

In [24]:
root = bst_from_order([4, 2, 5, 1, 3])
head = wrong_no_close(root)
print("tail.right =", head.right.right.right.right.right)    # None: đi 5 bước ra None
print(check_circular(head, [1, 2, 3, 4, 5]))

tail.right = None
(False, 'right/left không đối xứng tại nút 5')


**Vì sao sai:** đề yêu cầu *vòng tròn*: `1.left` phải là `5` và `5.right` phải là `1`. **Sửa:** sau khi duyệt xong thêm `prev.right = head; head.left = prev` (xem Cách 2).

### Lỗi 2: Quên `nonlocal`

In [25]:
def wrong_nonlocal(root):
    head = prev = None
    def dfs(node):
        if node is None:
            return
        dfs(node.left)
        if prev is not None:       # đọc prev -> Python coi prev là biến cục bộ (vì có gán bên dưới)
            prev.right = node
        prev = node                # GÁN -> prev cục bộ chưa có giá trị khi đọc ở trên
        dfs(node.right)
    dfs(root)

try:
    wrong_nonlocal(build_tree([2, 1, 3]))
except UnboundLocalError as e:
    print("UnboundLocalError:", e)

UnboundLocalError: cannot access local variable 'prev' where it is not associated with a value


**Vì sao sai:** trong `dfs`, câu `prev = node` biến `prev` thành biến **cục bộ** của `dfs`, nên dòng `if prev is not None` đọc một biến chưa được gán. **Sửa:** khai báo `nonlocal head, prev` ở đầu `dfs` (hoặc dùng một list/đối tượng chứa trạng thái).

### Lỗi 3: Nối con trỏ **trước khi** đi xuống cây con trái (phá cây đang duyệt)

Nếu ta sửa `node.left = prev` rồi mới gọi `dfs(node.left)`, thì `node.left` không còn là cây con trái nữa.

In [26]:
def wrong_link_before(root, limit=50):
    prev = None
    steps = 0
    def dfs(node):
        nonlocal prev, steps
        steps += 1
        if node is None or steps > limit:      # limit chỉ để demo không treo máy
            return
        if prev is not None:
            node.left = prev                   # ghi đè con trái TRƯỚC khi đi xuống
            prev.right = node
        prev = node
        dfs(node.left)
        dfs(node.right)
    dfs(root)
    return steps

steps = wrong_link_before(bst_from_order([4, 2, 5, 1, 3]))
print("số bước:", steps, "(vượt giới hạn 50 -> nó lặp vòng trong cấu trúc bị ghi đè)")

số bước: 101 (vượt giới hạn 50 -> nó lặp vòng trong cấu trúc bị ghi đè)


**Vì sao sai:** đã nói ở mục 6, mỗi con trỏ chỉ được sửa **sau khi** dùng xong. Với thứ tự "nút → trái", `node.left` bị ghi đè khi chưa duyệt cây con trái, cây bị rối và đệ quy có thể **lặp vô hạn**. **Sửa:** dùng **inorder** (trái → nút → phải): sửa `node.left` *sau* `dfs(node.left)`.

### Lỗi 4: Trả về `root` thay vì nút nhỏ nhất, và quên cây rỗng

In [27]:
root = bst_from_order([4, 2, 5, 1, 3])
wrong = wrong_return_root(root)
print("trả về", wrong.val, "nhưng nút nhỏ nhất là 1")
print(check_circular(wrong, [1, 2, 3, 4, 5]))

def wrong_empty(root):
    head = prev = None
    # ... (duyệt)
    # prev là None khi cây rỗng -> prev.right = head gây AttributeError
    prev.right = head

try:
    wrong_empty(None)
except AttributeError as e:
    print("AttributeError:", e)

trả về 4 nhưng nút nhỏ nhất là 1
(False, 'thứ tự sai: [4, 5, 1, 2, 3]')
AttributeError: 'NoneType' object has no attribute 'right' and no __dict__ for setting new attributes


**Sửa:** trả về `head` (nút nhỏ nhất, thăm đầu tiên). Và **luôn** có `if root is None: return None` ở đầu, vì `prev`/`head` sẽ là `None` nếu cây rỗng.

## 14. Mẹo & nhận dạng pattern

- **BST + "sắp xếp" ⇒ nghĩ ngay tới inorder.** Hễ đề nhắc BST và thứ tự tăng dần, hãy thử inorder.
- **"Nối nút hiện tại với nút trước đó" ⇒ biến `prev`.** Mẫu này lặp lại ở nhiều bài cây: *Validate BST (98)*, *Recover BST (99)*, *Minimum Absolute Difference in BST (530)*, *Increasing Order Search Tree (897)*.
- **"Tại chỗ" trên cây/danh sách ⇒ đổi nghĩa con trỏ, không tạo nút mới.** Kiểm tra bằng `id()` của nút trước/sau.
- **Hai đầu của danh sách vòng ⇒ gán sau cùng**, khi đã biết `head` và `tail`.
- **Cây có thể sâu ⇒ cân nhắc inorder lặp** bằng stack (hoặc Morris) để không dính giới hạn đệ quy.
- Muốn "kiểm tra đúng" một bài trả về **cấu trúc con trỏ**: viết **giám khảo** (`check_circular`) đi cả hai chiều và kiểm tra tính đối xứng `x.right.left is x`.

## 15. Biến thể

1. **Danh sách đôi không vòng:** bỏ bước khép vòng, trả `head` với `head.left = None`, `tail.right = None`.
2. **Increasing Order Search Tree (897):** chỉ dùng `right` (`left = None`), thành chuỗi đơn.
3. **Flatten Binary Tree to Linked List (114):** theo **tiền thứ tự**, không phải trung thứ tự.
4. **Cây nhị phân bất kỳ** (không phải BST) thành danh sách đôi theo thứ tự inorder: giải thuật y hệt (chỉ đổi "đã sắp xếp" thành "theo inorder").
5. **Chiều ngược:** *Convert Sorted List to BST (109)* / *Convert Sorted Array to BST (108)* — dựng lại cây cân bằng từ danh sách đã sắp xếp.
6. **Trả thêm phần tử lớn nhất** (tail) hay chiều dài danh sách: `tail = head.left`.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. List các nút | gom inorder vào list rồi nối `i-1`, `(i+1)%n` | `O(n)` | `O(n)` | hiểu đề, oracle |
| 2. Inorder + `prev` | nối nút vừa thăm với `prev`, cuối cùng khép vòng | `O(n)` | `O(h)` | **nên nộp** |
| 3. Chia để trị | vòng con trái + gốc + vòng con phải bằng `concat` | `O(n)` | `O(h)` | tư duy đệ quy "hàm tự đủ nghĩa" |
| 4. Inorder lặp | stack tường minh | `O(n)` | `O(h)` | cây sâu, tránh RecursionError |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: (a) `inorder_vals(build_tree([5, 3, 8, 1, 4]))`; (b) với cây trên sau `Solution().treeToDoublyList`, `head.val`, `head.left.val`, `head.right.val`; (c) kết quả `Solution().treeToDoublyList(None)`; (d) với cây một nút `Node(9)` thì `head.right is head`?

**Bài 2 (tính tay).** Với cây `[4,2,6,1,3,5,7]` (cây đầy đủ 7 nút) hãy lập bảng như mục 4: thứ tự thăm, `prev`, và hai con trỏ được gán ở mỗi lần thăm; cuối cùng hai con trỏ khép vòng.

**Bài 3 (điền TODO).** Hoàn thành `tree_to_dll_todo(root)` theo Cách 2.

In [28]:
def tree_to_dll_todo(root):
    # TODO 1: nếu root là None thì trả về None
    head = prev = None

    def dfs(node):
        # TODO 2: khai báo nonlocal cho head, prev
        # TODO 3: nếu node là None thì return
        # TODO 4: dfs(node.left)
        # TODO 5: nếu prev tồn tại thì nối prev.right = node, node.left = prev; ngược lại head = node
        # TODO 6: prev = node; rồi dfs(node.right)
        pass

    # TODO 7: gọi dfs(root), khép vòng prev.right = head; head.left = prev, rồi trả về head
    return None

# KHÔNG gọi hàm TODO ở đây để Run All không lỗi (TODO chưa điền).

**Bài 4 (tự viết lại).** Viết `tree_to_dll_two_pass(root)` làm theo **hai bước rõ ràng**: (1) một hàm đệ quy `collect(node, out)` thêm các nút theo inorder vào `out`; (2) một vòng `for` nối `out[i]` với `out[i+1]` (**không** dùng `%`), rồi khép vòng thủ công bằng `out[-1]` và `out[0]`. Kiểm tra bằng `check_circular`.

**Bài 5 (biến thể / mini challenge).** (a) Viết `increasing_bst(root)` (bài 897): trả về gốc mới là nút nhỏ nhất, mọi `left = None`, mỗi nút chỉ có con phải là nút lớn kế tiếp (không vòng). (b) Viết `ring_length(head)` đếm số nút trong danh sách vòng (đi `right` đến khi quay về `head`), và kiểm tra `ring_length(Solution().treeToDoublyList(build_tree([4,2,5,1,3]))) == 5`.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: inorder luôn tăng dần: `[1, 3, 4, 5, 8]`; `head` là nút nhỏ nhất, `head.left` là nút lớn nhất; cây một nút tự trỏ chính nó. Bài 3: nhớ `nonlocal`, nhớ `return None` cho cây rỗng.
- **Mức 2 (vừa):** Bài 2: thứ tự thăm `1,2,3,4,5,6,7`; ở lần thăm `k > 1` gán `(k-1).right = k` và `k.left = k-1`; khép vòng `7.right = 1`, `1.left = 7`. Bài 4: `for i in range(len(out) - 1): out[i].right = out[i+1]; out[i+1].left = out[i]`.
- **Mức 3 (sâu):** Bài 5(a): trong inorder, thay vì `node.left = prev` hãy đặt `node.left = None` và `prev.right = node`; nút nhỏ nhất đầu tiên là gốc mới (dùng `dummy` để khỏi xử lý riêng). 5(b): `n = 1; cur = head.right; while cur is not head: n += 1; cur = cur.right`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [29]:
# Bài 1
assert inorder_vals(build_tree([5, 3, 8, 1, 4])) == [1, 3, 4, 5, 8]
h = Solution().treeToDoublyList(build_tree([5, 3, 8, 1, 4]))
assert (h.val, h.left.val, h.right.val) == (1, 8, 3)
assert Solution().treeToDoublyList(None) is None
one = Node(9)
h1 = Solution().treeToDoublyList(one)
assert h1.right is h1

# Bài 2: thứ tự thăm 1..7 và vòng đúng
h2 = Solution().treeToDoublyList(build_tree([4, 2, 6, 1, 3, 5, 7]))
assert check_circular(h2, [1, 2, 3, 4, 5, 6, 7]) == (True, "ok")
assert h2.left.val == 7 and h2.left.right is h2


# Bài 3
def tree_to_dll_todo_answer(root):
    if root is None:
        return None
    head = prev = None

    def dfs(node):
        nonlocal head, prev
        if node is None:
            return
        dfs(node.left)
        if prev is not None:
            prev.right = node
            node.left = prev
        else:
            head = node
        prev = node
        dfs(node.right)

    dfs(root)
    prev.right = head
    head.left = prev
    return head

h3 = tree_to_dll_todo_answer(build_tree([4, 2, 5, 1, 3]))
assert check_circular(h3, [1, 2, 3, 4, 5]) == (True, "ok")
assert tree_to_dll_todo_answer(None) is None


# Bài 4
def tree_to_dll_two_pass(root):
    if root is None:
        return None
    out = []

    def collect(node, out):
        if node is None:
            return
        collect(node.left, out)
        out.append(node)
        collect(node.right, out)

    collect(root, out)
    for i in range(len(out) - 1):
        out[i].right = out[i + 1]
        out[i + 1].left = out[i]
    out[-1].right = out[0]
    out[0].left = out[-1]
    return out[0]

for order in ([4, 2, 5, 1, 3], [1], [3, 2, 1], [1, 2, 3]):
    assert check_circular(tree_to_dll_two_pass(bst_from_order(order)), sorted(order)) == (True, "ok")
assert tree_to_dll_two_pass(None) is None


# Bài 5
def increasing_bst(root):
    dummy = Node(0)
    prev = dummy

    def dfs(node):
        nonlocal prev
        if node is None:
            return
        dfs(node.left)
        node.left = None
        prev.right = node
        prev = node
        dfs(node.right)

    dfs(root)
    return dummy.right

r5 = increasing_bst(build_tree([5, 3, 6, 2, 4, None, 8, 1, None, None, None, 7, 9]))
vals, cur = [], r5
while cur:
    assert cur.left is None
    vals.append(cur.val); cur = cur.right
assert vals == [1, 2, 3, 4, 5, 6, 7, 8, 9]


def ring_length(head):
    if head is None:
        return 0
    n, cur = 1, head.right
    while cur is not head:
        n += 1
        cur = cur.right
    return n

assert ring_length(Solution().treeToDoublyList(build_tree([4, 2, 5, 1, 3]))) == 5
assert ring_length(None) == 0
print("Đáp án bài tập: OK")

Đáp án bài tập: OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (inorder + `prev`, rồi khép vòng)

```python
class Solution:
    def treeToDoublyList(self, root: 'Optional[Node]') -> 'Optional[Node]':
        def dfs(node):
            nonlocal prev, head
            if node is None:
                return
            dfs(node.left)
            if prev:
                prev.right = node
                node.left = prev
            else:
                head = node
            prev = node
            dfs(node.right)

        if root is None:
            return None
        head = prev = None
        dfs(root)
        prev.right = head
        head.left = prev
        return head
```

**Checklist**

- [ ] Có `if root is None: return None` **trước** khi dùng `prev`/`head`?
- [ ] Duyệt đúng **inorder** (trái → nút → phải), và chỉ sửa con trỏ của nút **sau khi** đã đi xong cây con trái?
- [ ] Khai báo `nonlocal head, prev` (hoặc dùng cách khác) trong hàm đệ quy lồng?
- [ ] Gán `head` đúng một lần (khi `prev` còn `None`)?
- [ ] **Khép vòng** hai chiều: `prev.right = head` **và** `head.left = prev`?
- [ ] Trả về `head` (nút nhỏ nhất), không phải `root`?
- [ ] Không khai báo lại `Node` trong bài nộp; đã thử rỗng, một nút, cây lệch, cây đầy đủ?
- [ ] Nếu lo `RecursionError` (cây lệch 2000 nút): đã cân nhắc bản lặp bằng stack?